# **Garbage Image Processing**
- Muhammad Alfi Syahrin
- Muhammad Nadhim Abdirrahman
- Syahrul Reza Ananda

## Preparation

In [1]:
import os
import shutil as sh
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Reshape, LSTM, Bidirectional
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Download dataset
!kaggle datasets download farzadnekouei/trash-type-image-dataset

In [8]:
# Unzip dataset
sh.unpack_archive("trash-type-image-dataset.zip")

## Bagi Data

In [2]:
# Define dataset directory
dataset_dir = "TrashType_Image_Dataset"
train_dir = "dataset/train"
val_dir = "dataset/val"

In [3]:
# Create train and val directories
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

In [3]:
# Get list of object folders
object_folders = [f for f in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, f))]

In [4]:
object_folders

['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

In [5]:
# Resize dimensions
IMG_HEIGHT = 224
IMG_WIDTH = 224

In [20]:
# Collect all image paths and their corresponding labels
image_paths = []
labels = []

for obj_folder in object_folders:
    obj_path = os.path.join(dataset_dir, obj_folder)
    for img in os.listdir(obj_path):
        if img.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(obj_path, img))
            labels.append(obj_folder)  # Use folder name as label

# Split into train and validation with stratification
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# Function to resize and save images
def resize_and_save(img_list, labels_list, dest_dir):
    for img_path, label in zip(img_list, labels_list):
        dest_label_dir = os.path.join(dest_dir, label)
        os.makedirs(dest_label_dir, exist_ok=True)
        dest_img_path = os.path.join(dest_label_dir, os.path.basename(img_path))
        with Image.open(img_path) as image:
            resized_image = image.resize((IMG_WIDTH, IMG_HEIGHT))
            resized_image.save(dest_img_path)

# Resize and save images to train and validation directories
resize_and_save(train_paths, train_labels, train_dir)
resize_and_save(val_paths, val_labels, val_dir)

print(f"Data has been split, resized, and organized into:\nTrain directory: {train_dir}\nVal directory: {val_dir}")

Data has been split, resized, and organized into:
Train directory: dataset/train
Val directory: dataset/val


## Training

In [6]:
# Define image dimensions and parameters
BATCH_SIZE = 32
NUM_CLASSES = len(object_folders)  # Change this according to the number of classes

In [7]:
# Data augmentation and preprocessing
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1.0 / 255)

# Load data
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

validation_generator = validation_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)


Found 2021 images belonging to 6 classes.
Found 506 images belonging to 6 classes.


In [8]:
# Modify the CNN model to include BILSTM
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_WIDTH, IMG_HEIGHT, 3)),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    Conv2D(256, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    # Flatten the convolutional output
    Flatten(),

    # Reshape the flat output to (timesteps, features) for LSTM
    Reshape((1, -1)),  # Reshape to a single timestep with features equal to the flattened size

    # Add BILSTM layer
    Bidirectional(LSTM(128, return_sequences=False)),

    # Fully connected layers
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

c:\Users\syahr\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
# Display the model architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 1, 36864)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │    37,880,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,302,918 (146.11 MB)

 Trainable params: 38,302,918 (146.11 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [12]:
# Train the model
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    validation_steps=None,
    epochs=50,
    callbacks=[early_stopping]
)

Epoch 1/50


c:\Users\syahr\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


64/64 ━━━━━━━━━━━━━━━━━━━━ 47s 678ms/step - accuracy: 0.2376 - loss: 1.7447 - val_accuracy: 0.3518 - val_loss: 1.5963
Epoch 2/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 44s 673ms/step - accuracy: 0.3251 - loss: 1.5465 - val_accuracy: 0.4427 - val_loss: 1.3536
Epoch 3/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 44s 672ms/step - accuracy: 0.4322 - loss: 1.3549 - val_accuracy: 0.4921 - val_loss: 1.3313
Epoch 4/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 44s 671ms/step - accuracy: 0.4214 - loss: 1.4048 - val_accuracy: 0.4842 - val_loss: 1.2864
Epoch 5/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 44s 667ms/step - accuracy: 0.4449 - loss: 1.3569 - val_accuracy: 0.4783 - val_loss: 1.2952
Epoch 6/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 44s 672ms/step - accuracy: 0.4768 - loss: 1.2800 - val_accuracy: 0.5198 - val_loss: 1.2206
Epoch 7/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 44s 668ms/step - accuracy: 0.5120 - loss: 1.2391 - val_accuracy: 0.5395 - val_loss: 1.2252
Epoch 8/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 44s 668ms/step - accuracy: 0.5514 - loss: 1.1751 - val_accuracy: 0.585

In [13]:
# Evaluate the model
eval_result = model.evaluate(validation_generator)
print(f"Validation Accuracy: {eval_result[1] * 100:.2f}%")

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 125ms/step - accuracy: 0.7345 - loss: 0.7198
Validation Accuracy: 74.11%


In [14]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np


# Path ke file gambar
image_path = "cardboard.jpg"

# Muat gambar dengan ukuran yang sesuai untuk model
image = load_img(image_path, target_size=(224, 224))  # Ganti sesuai ukuran input model

# Ubah gambar menjadi array numpy
image_array = img_to_array(image)

# Normalisasi nilai pixel (rescale ke 0-1)
image_array = image_array / 255.0

# Tambahkan dimensi batch (1, 150, 150, 3)
image_array = np.expand_dims(image_array, axis=0)

In [15]:
# Lakukan prediksi
predictions = model.predict(image_array)

# Cetak hasil prediksi
print("Predictions:", predictions)

# Jika menggunakan label, ambil kelas dengan probabilitas tertinggi
predicted_class = np.argmax(predictions, axis=1)
print("Predicted class:", predicted_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
Predictions: [[9.7646183e-01 6.9443043e-04 4.7703772e-03 1.4654296e-02 8.8741060e-04
  2.5316507e-03]]
Predicted class: [0]


In [16]:
# Mapping indeks ke label kelas
class_indices = train_generator.class_indices
class_labels = {v: k for k, v in class_indices.items()}  # Membalik key dan value
print("Predicted label:", class_labels[predicted_class[0]])


Predicted label: cardboard


In [17]:
# Save the model
model.save("image_classification_model_2.h5")